# BHRAM-IL Colab Experiment Pipeline
This notebook loads the frozen final dataset, runs inference with Qwen/Qwen2.5-3B-Instruct, evaluates responses, computes metrics, and saves publication-ready tables and plots.

**Dataset:** `final4langv1.jsonl`
**Languages:** English, Hindi, Kannada, Tamil
**Categories:** chrono_questions, factual_questions, indian_questions
**Benchmark size:** 1200 rows, 300 aligned groups

## Cell 1: Install all dependencies

In [ ]:
!pip install -q transformers accelerate bitsandbytes sentencepiece protobuf pandas openpyxl tqdm numpy scipy "thefuzz[speedup]" rouge-score sacrebleu indic-transliteration fasttext-wheel

## Cell 2: Mount Google Drive

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

drive_root = Path('/content/drive/MyDrive')
results_root = drive_root / 'results' / 'BHRAM_IL'
results_root.mkdir(parents=True, exist_ok=True)
results_root

## Cell 3: Clone or load the BHRAM-IL repository

In [ ]:
from pathlib import Path
repo_dir = Path('/content/drive/MyDrive/BHRAM-IL')
if not repo_dir.exists():
    !git clone https://github.com/<YOUR_ORG>/BHRAM-IL.git "{repo_dir}"

%cd "{repo_dir}"
!find . -maxdepth 2 -type f | sort | head -n 20

## Cell 4: Load the final dataset and run sanity checks

In [ ]:
import json
from pathlib import Path
from collections import Counter

repo_dir = Path('/content/drive/MyDrive/BHRAM-IL')
dataset_path = repo_dir / 'dataset' / 'final_4lang_seed' / 'final4langv1.jsonl'
rows = [json.loads(line) for line in dataset_path.read_text(encoding='utf-8').splitlines() if line.strip()]
print('Dataset path:', dataset_path)
print('Total rows:', len(rows))
languages = sorted({row['language'] for row in rows})
categories = sorted({row['category'] for row in rows})
groups = len({(row['category'], row['question_id']) for row in rows})
print('Languages:', languages)
print('Categories:', categories)
print('Unique groups (category, question_id):', groups)
print('Rows per language:', Counter(row['language'] for row in rows))
print('Rows per category:', Counter(row['category'] for row in rows))

## Cell 5: Load the Hugging Face model with 4-bit quantization

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_name = 'Qwen/Qwen2.5-3B-Instruct'
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map='auto',
    quantization_config=bnb_config,
    trust_remote_code=True,
)
model.eval()
print('Loaded model:', model_name)
print('Device:', next(model.parameters()).device)

## Cell 6: Run inference with the modified generation script

In [ ]:
from pathlib import Path
repo_dir = Path('/content/drive/MyDrive/BHRAM-IL')
dataset_path = repo_dir / 'dataset' / 'final_4lang_seed' / 'final4langv1.jsonl'
output_file = Path('/content/drive/MyDrive/results/BHRAM_IL/generation/output.final4langv1.qwen2.5-3b-native.jsonl')
output_file.parent.mkdir(parents=True, exist_ok=True)
print('Output file:', output_file)
%cd "{repo_dir}/run"
!python generate_response_hf.py "-m" "Qwen/Qwen2.5-3B-Instruct" "-d" "{dataset_path}" "--output" "{output_file}" "--prompt-mode" "english" "--quantization" "4bit" "--trust-remote-code" "--max-new-tokens" "128" "--temperature" "0.0" "--top-p" "0.95"
%cd "{repo_dir}"


## Cell 7: Run BHRAM-IL evaluation

In [ ]:
from pathlib import Path
eval_output = Path('/content/drive/MyDrive/results/BHRAM_IL/eval/bhram_qwen_native.eval.jsonl')
eval_output.parent.mkdir(parents=True, exist_ok=True)
dataset_output = Path('/content/drive/MyDrive/results/BHRAM_IL/generation/output.final4langv1.qwen2.5-3b-native.jsonl')
print('Evaluation output:', eval_output)
!python evaluate/evaluation.py -i "{dataset_output}" -o "{eval_output}" --verbose

## Cell 8: Compute overall, per-language, and per-category metrics

In [ ]:
import json
import pandas as pd
from pathlib import Path

eval_output = Path('/content/drive/MyDrive/results/BHRAM_IL/eval/bhram_qwen_native.eval.jsonl')
rows = [json.loads(line) for line in eval_output.read_text(encoding='utf-8').splitlines() if line.strip()]
df = pd.json_normalize(rows)

def summarize(df):
    return pd.Series({
        'questions': len(df),
        'primary_score': df['scores.primary'].mean(),
        'fuzzy_score': df['scores.fuzzy'].mean(),
        'language_mismatch_rate': df['language_hallucination.is_hallucination'].mean(),
        'success_rate': df['match.primary'].mean(),
        'failure_rate': 1 - df['match.primary'].mean(),
    })

overall = summarize(df)
by_language = df.groupby('language').apply(summarize).reset_index()
by_category = df.groupby('category').apply(summarize).reset_index()

print('Overall metrics:')
print(overall.to_string())
print('
By language:')
print(by_language.to_string(index=False))
print('
By category:')
print(by_category.to_string(index=False))

## Cell 9: Generate publication-quality plots

In [ ]:
import json
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from pathlib import Path

root = Path('/content/drive/MyDrive/results/BHRAM_IL')
root.mkdir(parents=True, exist_ok=True)
eval_output = root / 'eval' / 'bhram_qwen_native.eval.jsonl'
gen_output = root / 'generation' / 'output.final4langv1.qwen2.5-3b-native.jsonl'
rows = [json.loads(line) for line in eval_output.read_text(encoding='utf-8').splitlines() if line.strip()]
df = pd.json_normalize(rows)
gen_rows = [json.loads(line) for line in gen_output.read_text(encoding='utf-8').splitlines() if line.strip()]
len_df = pd.DataFrame([
    {
        'question_id': row['question_id'],
        'language': row['language'],
        'category': row['category'],
        'response_length': len(row['response']['content']) if isinstance(row['response'], dict) else len(str(row['response'])),
    }
    for row in gen_rows
])

sns.set(style='whitegrid', font_scale=1.1)

fig1, ax1 = plt.subplots(figsize=(8, 5))
sns.barplot(x='language', y='scores.primary', data=df, estimator='mean', ax=ax1)
ax1.set_title('Primary Score by Language')
ax1.set_ylabel('Primary Score')
fig1.tight_layout()
fig1.savefig(root / 'primary_score_by_language.png', dpi=300)

fig2, ax2 = plt.subplots(figsize=(10, 5))
sns.barplot(x='category', y='scores.primary', data=df, estimator='mean', ax=ax2)
ax2.set_title('Primary Score by Category')
ax2.set_ylabel('Primary Score')
fig2.tight_layout()
fig2.savefig(root / 'primary_score_by_category.png', dpi=300)

heat = df.pivot_table(index='category', columns='language', values='scores.primary', aggfunc='mean')
fig3, ax3 = plt.subplots(figsize=(8, 6))
sns.heatmap(heat, annot=True, fmt='.3f', cmap='viridis', ax=ax3)
ax3.set_title('Primary Score Heatmap by Category and Language')
fig3.tight_layout()
fig3.savefig(root / 'primary_score_heatmap_by_language_category.png', dpi=300)

pie_data = df['language_hallucination.hallucination_type'].value_counts(normalize=True).reset_index()
fig4, ax4 = plt.subplots(figsize=(7, 7))
ax4.pie(pie_data['language_hallucination.hallucination_type'], labels=pie_data['index'], autopct='%1.1f%%', startangle=140)
ax4.set_title('Failure Type Distribution')
fig4.tight_layout()
fig4.savefig(root / 'failure_type_distribution.png', dpi=300)

fig5, ax5 = plt.subplots(figsize=(8, 5))
sns.histplot(len_df, x='response_length', bins=30, kde=True, ax=ax5)
ax5.set_title('Response Length Distribution')
ax5.set_xlabel('Response length (characters)')
fig5.tight_layout()
fig5.savefig(root / 'response_length_histogram.png', dpi=300)

print('Saved plots to', root)

## Cell 10: Perform qualitative error analysis

In [ ]:
import json
import pandas as pd
from pathlib import Path

eval_output = Path('/content/drive/MyDrive/results/BHRAM_IL/eval/bhram_qwen_native.eval.jsonl')
rows = [json.loads(line) for line in eval_output.read_text(encoding='utf-8').splitlines() if line.strip()]
df = pd.json_normalize(rows)
failures = df[(df['match.primary'] == False) | (df['language_hallucination.is_hallucination'] == True)].copy()
failures['failure_reason'] = failures.apply(lambda row: 'language_mismatch' if row['language_hallucination.is_hallucination'] else 'primary_mismatch', axis=1)
failures = failures.sort_values(['match.primary', 'scores.primary'], ascending=[True, True]).head(20)
analysis = failures[[
    'question_id', 'language', 'category', 'predicted', 'expected',
    'failure_reason', 'scores.primary', 'scores.fuzzy',
    'language_hallucination.predicted_language',
    'language_hallucination.reference_language'
]]
analysis

## Cell 11: Generate a paper-ready summary table

In [ ]:
import json
import pandas as pd
from pathlib import Path

eval_output = Path('/content/drive/MyDrive/results/BHRAM_IL/eval/bhram_qwen_native.eval.jsonl')
rows = [json.loads(line) for line in eval_output.read_text(encoding='utf-8').splitlines() if line.strip()]
df = pd.json_normalize(rows)
summary_table = df.groupby('language').apply(lambda g: pd.Series({
    'Questions': len(g),
    'Primary Score': g['scores.primary'].mean(),
    'Fuzzy Score': g['scores.fuzzy'].mean(),
    'Hallucination %': 100 * g['language_hallucination.is_hallucination'].mean(),
})).reset_index()
summary_table

## Cell 12: Export all results to Google Drive

In [ ]:
import json
import pandas as pd
from pathlib import Path

root = Path('/content/drive/MyDrive/results/BHRAM_IL')
export_dir = root / 'export'
export_dir.mkdir(parents=True, exist_ok=True)
eval_output = root / 'eval' / 'bhram_qwen_native.eval.jsonl'
rows = [json.loads(line) for line in eval_output.read_text(encoding='utf-8').splitlines() if line.strip()]
df = pd.json_normalize(rows)
summary_by_language = df.groupby('language').apply(lambda g: pd.Series({
    'Questions': len(g),
    'Primary Score': g['scores.primary'].mean(),
    'Fuzzy Score': g['scores.fuzzy'].mean(),
    'Hallucination %': 100 * g['language_hallucination.is_hallucination'].mean(),
})).reset_index()
summary_by_category = df.groupby('category').apply(lambda g: pd.Series({
    'Questions': len(g),
    'Primary Score': g['scores.primary'].mean(),
    'Fuzzy Score': g['scores.fuzzy'].mean(),
    'Hallucination %': 100 * g['language_hallucination.is_hallucination'].mean(),
})).reset_index()
summary_by_language.to_csv(export_dir / 'summary_by_language.csv', index=False)
summary_by_category.to_csv(export_dir / 'summary_by_category.csv', index=False)
df.to_json(export_dir / 'evaluation_results.json', orient='records', force_ascii=False, indent=2)
df.to_csv(export_dir / 'evaluation_results.csv', index=False)
final_stats = {
    'total_rows': len(df),
    'total_groups': len(df[['category', 'question_id']].drop_duplicates()),
    'overall_primary_score': df['scores.primary'].mean(),
    'overall_fuzzy_score': df['scores.fuzzy'].mean(),
    'overall_hallucination_rate': float(df['language_hallucination.is_hallucination'].mean()),
}
with open(export_dir / 'final_statistics.json', 'w', encoding='utf-8') as f:
    json.dump(final_stats, f, ensure_ascii=False, indent=2)

print('Exported results to', export_dir)